# Notebook 00 — End-to-End Pipeline Test

This notebook scans every Excel file in the samples folder, classifies each sheet, and shows whether it should be parsed into the normalized market dataset or treated as a non-core Family B sheet.

In [3]:
from pathlib import Path
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'samples').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise FileNotFoundError('Could not find the repository root from the current notebook location.')

repo_root = find_repo_root(Path.cwd().resolve())
samples_dir = repo_root / 'samples'
excel_files = sorted(samples_dir.glob('*.xlsx'))

print('Repository root:', repo_root)
print('Excel files found:')
for path in excel_files:
    print('-', path.name)

Repository root: /home/yass/Desktop/DSS_CMR
Excel files found:
- Compo_All_Indices_20260731_copy.xlsx
- Data_sheet_without_legend.xlsx
- Données Marché Boursier_Projet_IA_copy.xlsx


In [4]:
def detect_family(df: pd.DataFrame) -> str:
    head = df.iloc[:8, :].astype(str).fillna('')
    if head.apply(lambda col: col.str.contains('code isin', case=False).any()).any():
        return 'Family A'
    if head.iloc[:, 0].astype(str).str.strip().str.match(r'^[A-Z0-9\- /]+$').sum() >= 3:
        return 'Family B'
    return 'Unknown'

for workbook_path in excel_files:
    print(f'\n=== {workbook_path.name} ===')
    try:
        xls = pd.ExcelFile(workbook_path)
        for sheet_name in xls.sheet_names:
            raw = pd.read_excel(workbook_path, sheet_name=sheet_name, header=None)
            family = detect_family(raw)
            print(f'[{sheet_name}] -> {family}')
            if family == 'Family A':
                print('  -> parse and include in normalized market dataset')
            elif family == 'Family B':
                print('  -> validate and ignore for normalized market dataset')
            else:
                print('  -> unable to classify')
    except Exception as exc:
        print(f'Error reading workbook: {exc}')


=== Compo_All_Indices_20260731_copy.xlsx ===
[MASI] -> Family A
  -> parse and include in normalized market dataset
[Sector Indices] -> Family A
  -> parse and include in normalized market dataset
[MASI 20] -> Family A
  -> parse and include in normalized market dataset
[MASI ESG] -> Family A
  -> parse and include in normalized market dataset
[MASI Mid and Small Cap] -> Family A
  -> parse and include in normalized market dataset

=== Data_sheet_without_legend.xlsx ===
[Feuille 1] -> Family B
  -> validate and ignore for normalized market dataset

=== Données Marché Boursier_Projet_IA_copy.xlsx ===
[Data] -> Family A
  -> parse and include in normalized market dataset
[Cours] -> Family A
  -> parse and include in normalized market dataset
[Bid] -> Family A
  -> parse and include in normalized market dataset
[Ask] -> Family A
  -> parse and include in normalized market dataset
[Quantité MC] -> Family A
  -> parse and include in normalized market dataset
[Volume MC] -> Family A
  -> pa